# Lab 0-04: Selecting Between Two Tools

An agent can make more than one tool available to an LLM. The model uses the user's request and the tool descriptions to predict which approved tool best fits. The agent then checks and runs the selected Python function.

This notebook uses the Qwen model configured for this lab. Run [03_tools.ipynb](03_tools.ipynb) first so the single-tool workflow is familiar.

## What You Will Learn

- distinguish between an LLM selecting a tool and an agent running a tool
- explain why clear tool descriptions help an LLM select the appropriate tool
- follow complete addition and multiplication tool-use loops
- identify why an agent must validate a model's tool request before execution

## 1. Connect to the Configured Qwen Model

This notebook reads the model name and Ollama endpoint from this lab's .env file. Complete Lab 0-02 first so the configured endpoint is available.

In [ ]:
# json converts Qwen's text response into a Python dictionary.
import json
# Path helps the notebook find this lab's configuration file reliably.
from pathlib import Path

# dotenv_values reads MODEL and OLLAMA_BASE_URL from the lab-local .env file.
from dotenv import dotenv_values
# OpenAI is used here because Ollama offers an OpenAI-compatible connection.
from openai import OpenAI

# This notebook must run from the lab0_04_ai_agent folder.
LAB_NAME = "lab0_04_ai_agent"
lab_dir = Path.cwd().resolve()
if lab_dir.name != LAB_NAME:
    raise FileNotFoundError(f"Open this notebook from the {LAB_NAME} folder.")

# Each lab has its own .env so its model settings do not affect other labs.
env_path = lab_dir / ".env"
if not env_path.exists():
    raise FileNotFoundError(f"Expected {env_path}. Copy .env.example to .env first.")

# Read the two connection settings that the agent needs before it can ask Qwen.
config = dotenv_values(env_path)
model_name = config.get("MODEL")
ollama_base_url = config.get("OLLAMA_BASE_URL")
if not model_name or not ollama_base_url:
    raise ValueError("MODEL or OLLAMA_BASE_URL is missing from .env")

# Create a client object. The client sends prompts to Qwen; it does not run tools itself.
client = OpenAI(base_url=ollama_base_url, api_key="ollama")
print("Model from .env:", model_name)
print("Ollama endpoint:", ollama_base_url)

## 2. Define the Approved Tools

Both tools take two integers, but they have different purposes. The `@tool` decorator turns each typed, documented Python function into the same reusable `Tool` structure introduced in Section 4.2 of the previous notebook. The registry is the agent's approved list: Qwen may request only a tool listed here.

In [ ]:
# inspect reads information already attached to a Python function, such as its name and inputs.
import inspect
# Callable labels the stored Python function so the Tool structure is easier to read.
from collections.abc import Callable


# Purpose: turn a Python type such as int into readable text for a tool description.
def type_name(annotation) -> str:
    """Return a readable name for a Python type annotation."""
    return getattr(annotation, "__name__", str(annotation))


# Purpose: store one function together with the information an LLM needs to request it.
class Tool:
    """A reusable wrapper around one Python function."""

    def __init__(self, name: str, description: str, function: Callable, arguments: list[str]):
        self.name = name
        self.description = description
        self.function = function
        self.arguments = arguments

    # This method creates the text that will be included in Qwen's instructions.
    def to_string(self) -> str:
        """Return the description supplied to the LLM."""
        return (
            f"Name: {self.name}\n"
            f"Description: {self.description}\n"
            f"Arguments: {', '.join(self.arguments)}"
        )

    # This lets the agent use a Tool object like a normal function after approval.
    def __call__(self, *arguments, **keyword_arguments):
        """Run the wrapped function after the agent checks the request."""
        return self.function(*arguments, **keyword_arguments)


# Purpose: build a Tool automatically from a typed, documented Python function.
def tool(function: Callable) -> Tool:
    """Turn a typed, documented function into a Tool object."""
    # Read the function's parameters instead of rewriting them in a separate description.
    signature = inspect.signature(function)
    arguments = [
        f"{parameter.name}: {type_name(parameter.annotation)}"
        for parameter in signature.parameters.values()
    ]
    return Tool(
        name=function.__name__,
        description=inspect.getdoc(function) or "No description provided.",
        function=function,
        arguments=arguments,
    )


# @tool replaces this function with a Tool object while preserving its name and description.
@tool
def add_integers(a: int, b: int) -> int:
    """Add two integers."""
    return a + b


# The second Tool has the same structure but a different capability.
@tool
def multiply_integers(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b


# The agent, not the LLM, owns this approved-tool registry.
APPROVED_TOOLS = {
    add_integers.name: add_integers,
    multiply_integers.name: multiply_integers,
}
# Print the generated descriptions so students can see exactly what Qwen will receive.
for approved_tool in APPROVED_TOOLS.values():
    print(approved_tool.to_string())


## 3. Ask Qwen to Select a Tool

The agent gives Qwen descriptions of both approved tools and requires a JSON tool request. Qwen predicts the tool name and its arguments from the user's question. It does not call either Python function.

In [ ]:
# Purpose: turn the two Tool objects into the available-tool text supplied to Qwen.
tool_descriptions = '\n\n'.join(
    approved_tool.to_string() for approved_tool in APPROVED_TOOLS.values()
)
# These instructions constrain the model to select one approved name and provide matching inputs.
tool_instructions = (
    "You generate tool requests for an agent.\n\n"
    "Available tools:\n"
    + tool_descriptions
    + "\n\nReturn exactly one valid JSON object. Its name must be one of the available tools,"
    + " and its arguments must contain integer values for a and b.\n\n"
    + "Example: {\"name\": \"add_integers\", \"arguments\": {\"a\": 0, \"b\": 0}}\n"
    + "Do not include Markdown, explanations, or any text outside the JSON object."
)


# Purpose: ask Qwen for a tool request, then convert its JSON text into Python data.
def request_tool_from_qwen(user_question: str) -> tuple[str, dict]:
    """Ask Qwen for one JSON tool request and parse the returned text."""
    # Send the tool descriptions as persistent instructions and the question as user input.
    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": tool_instructions},
            {"role": "user", "content": user_question},
        ],
        temperature=0,
    )
    # Qwen returns text; the agent must inspect and parse that text before it can use it.
    raw_tool_request = response.choices[0].message.content
    if not raw_tool_request:
        raise ValueError("The model returned no tool-request text. Rerun the cell.")
    # json.loads changes valid JSON text into the dictionary used by the agent.
    try:
        tool_request = json.loads(raw_tool_request)
    except json.JSONDecodeError as error:
        raise ValueError("The model did not return valid JSON. Rerun the cell or check the tool instructions.") from error
    return raw_tool_request, tool_request


# Purpose: enforce the approved-tool list, then run the selected Tool object.
def run_selected_tool(tool_request: dict) -> int:
    """Check the requested tool name, then run the matching approved function."""
    # Read the name Qwen requested rather than allowing it to call Python directly.
    tool_name = tool_request.get("name")
    if tool_name not in APPROVED_TOOLS:
        raise ValueError(f"Requested tool is not approved: {tool_name}")
    # ** passes {"a": 18, "b": 7} as named Python inputs: a=18, b=7.
    return APPROVED_TOOLS[tool_name](**tool_request["arguments"])


# Purpose: show the complete loop so students can trace the model request to the tool result.
def demonstrate_tool_selection(user_question: str) -> None:
    """Show Qwen's request, the selected tool, and the agent's result."""
    raw_tool_request, tool_request = request_tool_from_qwen(user_question)
    tool_result = run_selected_tool(tool_request)
    print("User question:", user_question)
    print("\nRaw JSON from Qwen:")
    print(raw_tool_request)
    print("\nParsed request:", tool_request)
    print("Selected tool:", tool_request["name"])
    print("Tool result:", tool_result)

### 3.1 Addition Request

For this addition question, Qwen should request add_integers, and the agent should run that function.

In [ ]:
# Run the same agent workflow with an addition question.
demonstrate_tool_selection("What is 18 plus 7?")

### 3.2 Multiplication Request

For this multiplication question, Qwen should request multiply_integers, and the agent should run that function.

In [ ]:
# Run the same workflow again; only the user's question changes.
demonstrate_tool_selection("What is 6 times 9?")

## What To Notice

The LLM selected a tool request from descriptions supplied by the agent. The agent program—not Qwen—looked up the selected function in APPROVED_TOOLS and ran it.

> **Note:** This focused example checks only that the requested tool name is approved. A production agent must also validate argument names and types, permissions, data scope, and other boundaries before it runs code.

**Reflect.** Why did add_integers fit the first question and multiply_integers fit the second? What could happen if the agent ran a tool that Qwen named without checking the approved-tool registry?

Next, open [05_agent_walkthrough.ipynb](05_agent_walkthrough.ipynb) to compare a plain model with a bounded agent that uses approved case materials.